In [64]:
import os
import json
import math
import time
import copy
import random
from pathlib import Path

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset,Subset
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


DATA_ROOT = Path("./prepared_balanced_nilm")
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

APPLIANCE = "washing_machine"   # or "fridge", "dishwasher"
TARGET_BUILDING = "building_04"

TRAIN_BUILDINGS = ["building_01", "building_02"]
VAL_BUILDING = "building_03"

WINDOW_SIZE = 129
BATCH_SIZE = 256
NUM_WORKERS = 0

SOURCE_EPOCHS = 20
SOURCE_LR = 1e-4

CALIBRATION_MODE = "all"


PHASE1_EPOCHS = 4
PHASE2_EPOCHS = 4
PHASE1_LR = 1e-5
PHASE2_LR = 5e-6
EVAL_BATCH_SIZE = 2048

CALIBRATION_RATIO = 0.05

SOURCE_CHECKPOINT = CHECKPOINT_DIR / f"best_{APPLIANCE}_source_regressor.pt"

print("DEVICE:", DEVICE)
print("APPLIANCE:", APPLIANCE)
print("TARGET_BUILDING:", TARGET_BUILDING)
print("SOURCE_CHECKPOINT:", SOURCE_CHECKPOINT)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE: cuda
APPLIANCE: washing_machine
TARGET_BUILDING: building_04
SOURCE_CHECKPOINT: checkpoints\best_washing_machine_source_regressor.pt


In [66]:
def building_dir(building):
    return DATA_ROOT / building

def csv_path(building):
    return building_dir(building) / "prepared_timeseries.csv"

def metadata_path(building):
    return building_dir(building) / "metadata.json"

def center_path(building, appliance, kind):
    return building_dir(building) / f"{appliance}_{kind}_centers.npy"
AGGREGATE_COLUMN = "aggregate_norm"

def target_norm_column(appliance):
    return f"{appliance}_norm"

def target_raw_column(appliance):
    return f"{appliance}_raw"

def target_clipped_column(appliance):
    return f"{appliance}_clipped"

def target_weight_column(appliance):
    return f"{appliance}_weight"

def target_active_column(appliance):
    return f"{appliance}_active"

def target_event_column(appliance):
    return f"{appliance}_event"

In [67]:
def find_first_existing_column(columns, candidates):
    cols_lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

def infer_columns(df, appliance):
    agg_col = AGGREGATE_COLUMN
    tgt_col = target_norm_column(appliance)

    if agg_col not in df.columns:
        raise ValueError(f"Missing aggregate column: {agg_col}")

    if tgt_col not in df.columns:
        raise ValueError(f"Missing target column: {tgt_col}")

    return agg_col, tgt_col
def load_building_dataframe(building):
    df = pd.read_csv(csv_path(building))
    return df

def load_building_metadata(building):
    with open(metadata_path(building), "r", encoding="utf-8") as f:
        meta = json.load(f)
    return meta

def get_appliance_metadata(building, appliance):
    meta = load_building_metadata(building)

    if appliance in meta and isinstance(meta[appliance], dict):
        return meta[appliance]

    return meta
def make_eval_subset_loader(base_loader, max_samples=200000, seed=42):
    ds = base_loader.dataset
    n = len(ds)

    if n == 0:
        return base_loader

    if n <= max_samples:
        return base_loader

    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(n, size=max_samples, replace=False))
    subset = Subset(ds, idx)

    return DataLoader(
        subset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

In [68]:
def get_meta_value(meta, keys, default=None):
    for k in keys:
        if k in meta:
            return meta[k]
    return default

def inverse_target_transform_np(y_norm, meta):
    mean_ = float(get_meta_value(meta, ["target_mean", "mean", "y_mean"], 0.0))
    std_ = float(get_meta_value(meta, ["target_std", "std", "y_std"], 1.0))
    use_log = bool(get_meta_value(meta, ["target_log1p", "use_log", "log_target"], False))

    y_t = y_norm * std_ + mean_
    if use_log:
        y = np.expm1(y_t)
    else:
        y = y_t
    return np.maximum(y, 0.0)

def inverse_target_transform_torch(y_norm, meta):
    mean_ = float(get_meta_value(meta, ["target_mean", "mean", "y_mean"], 0.0))
    std_ = float(get_meta_value(meta, ["target_std", "std", "y_std"], 1.0))
    use_log = bool(get_meta_value(meta, ["target_log1p", "use_log", "log_target"], False))

    y_t = y_norm * std_ + std_ * 0 + mean_
    y_t = y_norm * std_ + mean_
    if use_log:
        y = torch.expm1(y_t)
    else:
        y = y_t
    return torch.clamp(y, min=0.0)

In [69]:
class NILMWindowDataset(Dataset):
    def __init__(self, aggregate, target, centers, window_size=129, sample_weights=None):
        self.aggregate = aggregate.astype(np.float32)
        self.target = target.astype(np.float32)
        self.centers = centers.astype(np.int64)
        self.window_size = window_size
        self.half = window_size // 2

        if sample_weights is None:
            self.sample_weights = np.ones(len(self.target), dtype=np.float32)
        else:
            self.sample_weights = sample_weights.astype(np.float32)

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        c = int(self.centers[idx])
        l = c - self.half
        r = c + self.half + 1

        x = self.aggregate[l:r]
        y = self.target[c]
        w = self.sample_weights[c]

        x = torch.tensor(x, dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(y, dtype=torch.float32)
        w = torch.tensor(w, dtype=torch.float32)
        return x, y, w

In [70]:
def make_sample_weights(target_array, power_weight=2.0):
    t = np.abs(target_array.astype(np.float32))
    return 1.0 + power_weight * t
def prepare_building_arrays(building, appliance):
    df = load_building_dataframe(building)
    agg_col, tgt_col = infer_columns(df, appliance)

    aggregate = df[agg_col].to_numpy(dtype=np.float32)
    target = df[tgt_col].to_numpy(dtype=np.float32)

    weight_col = target_weight_column(appliance)
    if weight_col in df.columns:
        sample_weights = df[weight_col].to_numpy(dtype=np.float32)
    else:
        sample_weights = np.ones(len(df), dtype=np.float32)

    balanced_centers = np.load(center_path(building, appliance, "balanced"))
    all_centers = np.load(center_path(building, appliance, "all"))

    return {
        "aggregate": aggregate,
        "target": target,
        "weights": sample_weights,
        "balanced_centers": balanced_centers,
        "all_centers": all_centers
    }

In [71]:
def make_source_loaders(appliance, batch_size=256, window_size=129):
    train_sets = []

    for b in TRAIN_BUILDINGS:
        data = prepare_building_arrays(b, appliance)
        ds = NILMWindowDataset(
            aggregate=data["aggregate"],
            target=data["target"],
            centers=data["balanced_centers"],
            window_size=window_size,
            sample_weights=data["weights"]
        )
        train_sets.append(ds)

    train_ds = ConcatDataset(train_sets)

    val_data = prepare_building_arrays(VAL_BUILDING, appliance)
    val_ds = NILMWindowDataset(
        aggregate=val_data["aggregate"],
        target=val_data["target"],
        centers=val_data["balanced_centers"],
        window_size=window_size,
        sample_weights=val_data["weights"]
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)

    return train_loader, val_loader

In [72]:

def make_target_loaders(
    appliance,
    target_building,
    calibration_ratio=0.10,
    batch_size=256,
    window_size=129,
    mode="mixed",
    mixed_balanced_fraction=0.5
):
    data = prepare_building_arrays(target_building, appliance)

    aggregate = data["aggregate"]
    target = data["target"]
    weights = data["weights"]

    balanced = np.load(center_path(target_building, appliance, "balanced")).astype(np.int64)
    all_centers = np.load(center_path(target_building, appliance, "all")).astype(np.int64)
    inactive = np.load(center_path(target_building, appliance, "inactive")).astype(np.int64)

    rng = np.random.default_rng(SEED)

    if mode == "balanced":
        pool = balanced.copy()
        rng.shuffle(pool)
        n_cal = max(1, int(len(pool) * calibration_ratio))
        cal_centers = np.sort(pool[:n_cal])

    elif mode == "all":
        pool = all_centers.copy()
        rng.shuffle(pool)
        n_cal = max(1, int(len(pool) * calibration_ratio))
        cal_centers = np.sort(pool[:n_cal])

    elif mode == "mixed":
        total_n = max(1, int(len(all_centers) * calibration_ratio))

        n_bal = int(total_n * mixed_balanced_fraction)
        n_inactive = total_n - n_bal

        bal_pool = balanced.copy()
        inact_pool = inactive.copy()

        rng.shuffle(bal_pool)
        rng.shuffle(inact_pool)

        n_bal = min(n_bal, len(bal_pool))
        n_inactive = min(n_inactive, len(inact_pool))

        cal_bal = bal_pool[:n_bal]
        cal_inactive = inact_pool[:n_inactive]

        cal_centers = np.unique(np.concatenate([cal_bal, cal_inactive])).astype(np.int64)
        cal_centers = np.sort(cal_centers)

    else:
        raise ValueError("mode must be one of: 'balanced', 'all', 'mixed'")

    cal_set = set(cal_centers.tolist())

    balanced_holdout_centers = np.array(
        [c for c in balanced if int(c) not in cal_set],
        dtype=np.int64
    )

    full_holdout_centers = np.array(
        [c for c in all_centers if int(c) not in cal_set],
        dtype=np.int64
    )

    cal_ds = NILMWindowDataset(
        aggregate=aggregate,
        target=target,
        centers=cal_centers,
        window_size=window_size,
        sample_weights=weights
    )

    balanced_holdout_ds = NILMWindowDataset(
        aggregate=aggregate,
        target=target,
        centers=balanced_holdout_centers,
        window_size=window_size,
        sample_weights=weights
    )

    full_holdout_ds = NILMWindowDataset(
        aggregate=aggregate,
        target=target,
        centers=full_holdout_centers,
        window_size=window_size,
        sample_weights=weights
    )

    cal_loader = DataLoader(
        cal_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    balanced_holdout_loader = DataLoader(
        balanced_holdout_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    full_holdout_loader = DataLoader(
        full_holdout_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    print("=" * 80)
    print("TARGET SPLIT SUMMARY")
    print("=" * 80)
    print("Mode:", mode)
    print("Calibration ratio:", calibration_ratio)
    print("Calibration samples:", len(cal_centers))
    print("Balanced holdout samples:", len(balanced_holdout_centers))
    print("Full holdout samples:", len(full_holdout_centers))

    return cal_loader, balanced_holdout_loader, full_holdout_loader



In [73]:
class HybridCNNTransformer(nn.Module):
    def __init__(self, input_dim=1, cnn_channels=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(32, cnn_channels, kernel_size=5, padding=2),
            nn.ReLU()
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cnn_channels,
            nhead=nhead,
            dim_feedforward=128,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.regressor = nn.Sequential(
            nn.Linear(cnn_channels, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)
        x = self.transformer_encoder(x)
        x = x.transpose(1, 2)
        x = self.pool(x).squeeze(-1)
        y = self.regressor(x).squeeze(-1)
        return y

In [74]:
class WeightedAsymmetricHuberLoss(nn.Module):
    def __init__(self, delta=1.0, underpredict_weight=2.0):
        super().__init__()
        self.delta = delta
        self.underpredict_weight = underpredict_weight

    def forward(self, pred, target, sample_weight=None):
        err = pred - target
        abs_err = torch.abs(err)

        huber = torch.where(
            abs_err <= self.delta,
            0.5 * err ** 2,
            self.delta * (abs_err - 0.5 * self.delta)
        )

        asym = torch.where(pred < target, self.underpredict_weight, 1.0)
        loss = huber * asym

        if sample_weight is not None:
            loss = loss * sample_weight

        return loss.mean()

In [75]:
def mae_np(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def r2_np(y_true, y_pred):
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    if ss_tot == 0:
        return 0.0
    return 1.0 - ss_res / ss_tot

def evaluate(model, loader, device, meta):
    model.eval()
    preds, trues = [], []

    with torch.inference_mode():
        for x, y, w in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            pred = model(x)

            pred_raw = inverse_target_transform_torch(pred, meta).cpu().numpy()
            true_raw = inverse_target_transform_torch(y, meta).cpu().numpy()

            preds.append(pred_raw)
            trues.append(true_raw)

    if len(preds) == 0 or len(trues) == 0:
        return {
            "mae": None,
            "r2": None,
            "rel_mae_pct": None,
            "pred_mean": None,
            "true_mean": None,
            "num_samples": 0
        }

    preds = np.concatenate(preds)
    trues = np.concatenate(trues)

    mae = float(np.mean(np.abs(trues - preds)))
    ss_res = float(np.sum((trues - preds) ** 2))
    ss_tot = float(np.sum((trues - np.mean(trues)) ** 2))
    r2 = 0.0 if ss_tot == 0 else 1.0 - ss_res / ss_tot

    pred_mean = float(np.mean(preds))
    true_mean = float(np.mean(trues))
    rel_mae_pct = float(100.0 * mae / max(true_mean, 1e-8))

    return {
        "mae": mae,
        "r2": r2,
        "rel_mae_pct": rel_mae_pct,
        "pred_mean": pred_mean,
        "true_mean": true_mean,
        "num_samples": len(trues)
    }

In [76]:
def collect_predictions(model, loader, device, meta):
    model.eval()
    preds, trues = [], []

    with torch.inference_mode():
        for x, y, w in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            pred = model(x)

            pred_raw = inverse_target_transform_torch(pred, meta).cpu().numpy()
            true_raw = inverse_target_transform_torch(y, meta).cpu().numpy()

            preds.append(pred_raw)
            trues.append(true_raw)

    if len(preds) == 0:
        return None, None

    return np.concatenate(preds), np.concatenate(trues)


def regression_metrics(y_true, y_pred):
    if y_true is None or len(y_true) == 0:
        return {
            "mae": None,
            "r2": None,
            "rel_mae_pct": None,
            "pred_mean": None,
            "true_mean": None,
            "num_samples": 0
        }

    mae = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = 0.0 if ss_tot == 0 else 1.0 - ss_res / ss_tot

    pred_mean = float(np.mean(y_pred))
    true_mean = float(np.mean(y_true))
    rel_mae_pct = float(100.0 * mae / max(true_mean, 1e-8))

    return {
        "mae": mae,
        "r2": r2,
        "rel_mae_pct": rel_mae_pct,
        "pred_mean": pred_mean,
        "true_mean": true_mean,
        "num_samples": len(y_true)
    }


def apply_threshold(y_pred, threshold):
    y_pred_thr = y_pred.copy()
    y_pred_thr[y_pred_thr < threshold] = 0.0
    return y_pred_thr


def binary_state_metrics(y_true, y_pred, on_threshold_true=10.0, on_threshold_pred=10.0):
    true_on = (y_true >= on_threshold_true).astype(np.int32)
    pred_on = (y_pred >= on_threshold_pred).astype(np.int32)

    tp = int(np.sum((true_on == 1) & (pred_on == 1)))
    tn = int(np.sum((true_on == 0) & (pred_on == 0)))
    fp = int(np.sum((true_on == 0) & (pred_on == 1)))
    fn = int(np.sum((true_on == 1) & (pred_on == 0)))

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy
    }

def threshold_sweep(model, loader, device, meta,
                    thresholds=(0, 5, 10, 20, 30, 50, 75, 100),
                    on_threshold_true=10.0,
                    on_threshold_pred=None):
    y_pred, y_true = collect_predictions(model, loader, device, meta)

    if y_pred is None or y_true is None:
        return pd.DataFrame(columns=[
            "threshold", "mae", "r2", "rel_mae_pct",
            "pred_mean", "true_mean",
            "precision", "recall", "f1", "accuracy"
        ])

    rows = []

    for thr in thresholds:
        pred_thr = apply_threshold(y_pred, thr)

        reg = regression_metrics(y_true, pred_thr)
        state = binary_state_metrics(
            y_true,
            pred_thr,
            on_threshold_true=on_threshold_true,
            on_threshold_pred=thr if on_threshold_pred is None else on_threshold_pred
        )

        rows.append({
            "threshold": thr,
            "mae": reg["mae"],
            "r2": reg["r2"],
            "rel_mae_pct": reg["rel_mae_pct"],
            "pred_mean": reg["pred_mean"],
            "true_mean": reg["true_mean"],
            "precision": state["precision"],
            "recall": state["recall"],
            "f1": state["f1"],
            "accuracy": state["accuracy"]
        })

    return pd.DataFrame(rows)

def evaluate_thresholded(model, loader, device, meta,
                         pred_threshold=0.0,
                         on_threshold_true=10.0,
                         on_threshold_pred=10.0):
    y_pred, y_true = collect_predictions(model, loader, device, meta)

    if y_pred is None:
        return {
            "regression": {
                "mae": None,
                "r2": None,
                "rel_mae_pct": None,
                "pred_mean": None,
                "true_mean": None,
                "num_samples": 0
            },
            "state": {
                "tp": 0, "tn": 0, "fp": 0, "fn": 0,
                "precision": None, "recall": None, "f1": None, "accuracy": None
            }
        }

    y_pred_thr = apply_threshold(y_pred, pred_threshold)

    reg = regression_metrics(y_true, y_pred_thr)
    state = binary_state_metrics(
        y_true, y_pred_thr,
        on_threshold_true=on_threshold_true,
        on_threshold_pred=on_threshold_pred
    )

    return {
        "regression": reg,
        "state": state
    }

In [77]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    losses = []

    for x, y, w in loader:
        x = x.to(device)
        y = y.to(device)
        w = w.to(device)

        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y, w)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses))

def set_trainable_layers(model, mode="head"):
    for p in model.parameters():
        p.requires_grad = False

    if mode == "head":
        for p in model.regressor.parameters():
            p.requires_grad = True
    elif mode == "head_last_block":
        for p in model.regressor.parameters():
            p.requires_grad = True
        for p in model.transformer_encoder.layers[-1].parameters():
            p.requires_grad = True
    elif mode == "all":
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f"Unknown mode: {mode}")

def get_trainable_params(model):
    return [p for p in model.parameters() if p.requires_grad]

def save_checkpoint(model, path):
    torch.save(model.state_dict(), str(path))

def load_checkpoint_if_exists(model, path, device):
    if Path(path).exists():
        model.load_state_dict(torch.load(str(path), map_location=device))
        print(f"Loaded source model from: {path}")
        return True
    else:
        print(f"Pretrained model not found: {path}")
        return False

In [78]:
def train_source_model(model, train_loader, val_loader, loss_fn, device, meta, epochs=20, lr=1e-4, save_path=None):
    print("=" * 80)
    print(f"SOURCE TRAINING FOR {APPLIANCE}")
    print("=" * 80)

    set_trainable_layers(model, mode="all")
    optimizer = AdamW(get_trainable_params(model), lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_r2 = -1e18
    best_state = None

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_metrics = evaluate(model, val_loader, device, meta)
        scheduler.step(val_metrics["r2"])

        print(
            f"[Source {epoch:02d}/{epochs}] "
            f"loss={train_loss:.4f} | "
            f"val_mae={val_metrics['mae']:.4f} "
            f"val_r2={val_metrics['r2']:.4f} "
            f"pred_mean={val_metrics['pred_mean']:.4f} "
            f"true_mean={val_metrics['true_mean']:.4f} "
            f"lr={optimizer.param_groups[0]['lr']:.6f} | "
            f"time={time.time()-t0:.1f}s"
        )

        if val_metrics["r2"] > best_r2:
            best_r2 = val_metrics["r2"]
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
        if save_path is not None:
            save_checkpoint(model, save_path)
            print(f"Saved best source checkpoint to: {save_path}")

    return model

In [79]:
def calibrate_model(model, cal_loader, eval_loader, loss_fn, device, meta,
                    phase1_epochs=8, phase2_epochs=8, phase1_lr=5e-5, phase2_lr=2.5e-5):
    print("=" * 80)
    print("CALIBRATION")
    print("=" * 80)

    print("Calibration dataset size:", len(cal_loader.dataset))
    print("Eval dataset size       :", len(eval_loader.dataset))

    best_r2 = -1e18
    best_state = None

    # -------------------------
    # Phase 1: train head only
    # -------------------------
    set_trainable_layers(model, mode="head")
    optimizer = AdamW(get_trainable_params(model), lr=phase1_lr, weight_decay=1e-4)

    for epoch in range(1, phase1_epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, cal_loader, optimizer, loss_fn, device)
        metrics = evaluate(model, eval_loader, device, meta)

        if metrics["num_samples"] == 0:
            print(
                f"[Phase1 {epoch:02d}/{phase1_epochs}] "
                f"loss={train_loss:.4f} | "
                f"eval skipped (empty loader) | "
                f"lr={optimizer.param_groups[0]['lr']:.6f} | "
                f"time={time.time()-t0:.1f}s"
            )
            continue

        print(
            f"[Phase1 {epoch:02d}/{phase1_epochs}] "
            f"loss={train_loss:.4f} | "
            f"eval_mae={metrics['mae']:.4f} "
            f"eval_r2={metrics['r2']:.4f} "
            f"pred_mean={metrics['pred_mean']:.4f} "
            f"true_mean={metrics['true_mean']:.4f} "
            f"lr={optimizer.param_groups[0]['lr']:.6f} | "
            f"time={time.time()-t0:.1f}s"
        )

        if metrics["r2"] is not None and metrics["r2"] > best_r2:
            best_r2 = metrics["r2"]
            best_state = copy.deepcopy(model.state_dict())

    # ----------------------------------------
    # Phase 2: unfreeze last transformer block
    # ----------------------------------------
    set_trainable_layers(model, mode="head_last_block")
    optimizer = AdamW(get_trainable_params(model), lr=phase2_lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    for epoch in range(1, phase2_epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, cal_loader, optimizer, loss_fn, device)
        metrics = evaluate(model, eval_loader, device, meta)

        if metrics["num_samples"] == 0:
            print(
                f"[Phase2 {epoch:02d}/{phase2_epochs}] "
                f"loss={train_loss:.4f} | "
                f"eval skipped (empty loader) | "
                f"lr={optimizer.param_groups[0]['lr']:.6f} | "
                f"time={time.time()-t0:.1f}s"
            )
            continue

        scheduler.step(metrics["r2"])

        print(
            f"[Phase2 {epoch:02d}/{phase2_epochs}] "
            f"loss={train_loss:.4f} | "
            f"eval_mae={metrics['mae']:.4f} "
            f"eval_r2={metrics['r2']:.4f} "
            f"pred_mean={metrics['pred_mean']:.4f} "
            f"true_mean={metrics['true_mean']:.4f} "
            f"lr={optimizer.param_groups[0]['lr']:.6f} | "
            f"time={time.time()-t0:.1f}s"
        )

        if metrics["r2"] is not None and metrics["r2"] > best_r2:
            best_r2 = metrics["r2"]
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
    else:
        print("Warning: no valid evaluation metrics were produced during calibration.")

    return model

In [80]:
source_train_loader, source_val_loader = make_source_loaders(
    appliance=APPLIANCE,
    batch_size=BATCH_SIZE,
    window_size=WINDOW_SIZE
)

calibration_loader, balanced_holdout_loader, full_holdout_loader = make_target_loaders(
    appliance=APPLIANCE,
    target_building=TARGET_BUILDING,
    calibration_ratio=CALIBRATION_RATIO,
    batch_size=BATCH_SIZE,
    window_size=WINDOW_SIZE,
    mode=CALIBRATION_MODE,
    mixed_balanced_fraction=0.5
)

print("Calibration dataset size:", len(calibration_loader.dataset))
print("Balanced holdout size  :", len(balanced_holdout_loader.dataset))
print("Full holdout size      :", len(full_holdout_loader.dataset))

calibration_eval_loader = make_eval_subset_loader(
    full_holdout_loader,
    max_samples=200000,
    seed=SEED
)

print("Calibration eval subset size:", len(calibration_eval_loader.dataset))
source_meta = get_appliance_metadata(VAL_BUILDING, APPLIANCE)
target_meta = get_appliance_metadata(TARGET_BUILDING, APPLIANCE)

print("All loaders ready.")
print("Source meta keys:", list(source_meta.keys()) if isinstance(source_meta, dict) else source_meta)
print("Target meta keys:", list(target_meta.keys()) if isinstance(target_meta, dict) else target_meta)

TARGET SPLIT SUMMARY
Mode: all
Calibration ratio: 0.05
Calibration samples: 817219
Balanced holdout samples: 66040
Full holdout samples: 15527166
Calibration dataset size: 817219
Balanced holdout size  : 66040
Full holdout size      : 15527166
Calibration eval subset size: 200000
All loaders ready.
Source meta keys: ['building', 'length', 'window_size', 'clip_percentile', 'shared_appliances', 'aggregate_stats', 'targets']
Target meta keys: ['building', 'length', 'window_size', 'clip_percentile', 'shared_appliances', 'aggregate_stats', 'targets']


In [81]:
model = HybridCNNTransformer().to(DEVICE)
loss_fn = WeightedAsymmetricHuberLoss(delta=1.0, underpredict_weight=2.0)

loaded = load_checkpoint_if_exists(model, SOURCE_CHECKPOINT, DEVICE)

if not loaded:
    print("Starting from scratch: training source model first.")
    model = train_source_model(
        model=model,
        train_loader=source_train_loader,
        val_loader=source_val_loader,
        loss_fn=loss_fn,
        device=DEVICE,
        meta=source_meta,
        epochs=SOURCE_EPOCHS,
        lr=SOURCE_LR,
        save_path=SOURCE_CHECKPOINT
    )

model.load_state_dict(torch.load(str(SOURCE_CHECKPOINT), map_location=DEVICE))
print(f"Using source checkpoint: {SOURCE_CHECKPOINT}")

Loaded source model from: checkpoints\best_washing_machine_source_regressor.pt
Using source checkpoint: checkpoints\best_washing_machine_source_regressor.pt


C:\Users\wassi\AppData\Local\Temp\ipykernel_22848\2062410601.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(str(path), map_location=de

In [82]:
print("=" * 80)
print("SOURCE-ONLY TARGET RESULTS")
print("=" * 80)

src_bal = evaluate(model, balanced_holdout_loader, DEVICE, target_meta)
src_full = evaluate(model, full_holdout_loader, DEVICE, target_meta)

print("[BALANCED HOLDOUT]")
for k, v in src_bal.items():
    print(f"{k} : {v}")

print("\n[FULL HOLDOUT]")
for k, v in src_full.items():
    print(f"{k} : {v}")

SOURCE-ONLY TARGET RESULTS
[BALANCED HOLDOUT]
mae : 1.051405668258667
r2 : 0.3742781647923563
rel_mae_pct : 45.289330973313184
pred_mean : 2.441439390182495
true_mean : 2.32153058052063
num_samples : 66040

[FULL HOLDOUT]
mae : 0.3736983835697174
r2 : -0.26076883835748177
rel_mae_pct : 157.28989576715162
pred_mean : 0.5024858117103577
true_mean : 0.23758575320243835
num_samples : 15527166


In [83]:
model = calibrate_model(
    model=model,
    cal_loader=calibration_loader,
    eval_loader=calibration_eval_loader,
    loss_fn=loss_fn,
    device=DEVICE,
    meta=target_meta,
    phase1_epochs=PHASE1_EPOCHS,
    phase2_epochs=PHASE2_EPOCHS,
    phase1_lr=PHASE1_LR,
    phase2_lr=PHASE2_LR
)

CALIBRATION
Calibration dataset size: 817219
Eval dataset size       : 200000
[Phase1 01/4] loss=0.4813 | eval_mae=0.3105 eval_r2=-0.3346 pred_mean=0.4819 true_mean=0.2380 lr=0.000010 | time=76.7s
[Phase1 02/4] loss=0.3930 | eval_mae=0.3100 eval_r2=-0.3380 pred_mean=0.4865 true_mean=0.2380 lr=0.000010 | time=76.6s
[Phase1 03/4] loss=0.3794 | eval_mae=0.3098 eval_r2=-0.3345 pred_mean=0.4904 true_mean=0.2380 lr=0.000010 | time=76.5s
[Phase1 04/4] loss=0.3713 | eval_mae=0.3091 eval_r2=-0.3270 pred_mean=0.4925 true_mean=0.2380 lr=0.000010 | time=76.3s
[Phase2 01/4] loss=0.3555 | eval_mae=0.3073 eval_r2=-0.3133 pred_mean=0.5005 true_mean=0.2380 lr=0.000005 | time=132.5s
[Phase2 02/4] loss=0.3416 | eval_mae=0.2999 eval_r2=-0.2656 pred_mean=0.4946 true_mean=0.2380 lr=0.000005 | time=132.3s
[Phase2 03/4] loss=0.3317 | eval_mae=0.2966 eval_r2=-0.2608 pred_mean=0.4951 true_mean=0.2380 lr=0.000005 | time=132.6s
[Phase2 04/4] loss=0.3228 | eval_mae=0.2893 eval_r2=-0.2188 pred_mean=0.4885 true_mean

In [84]:
sweep_df = threshold_sweep(
    model=model,
    loader=calibration_eval_loader,
    device=DEVICE,
    meta=target_meta,
    thresholds=(0, 5, 10, 20, 30, 50, 75, 100),
    on_threshold_true=10.0
)

sweep_df

,threshold,mae,r2,rel_mae_pct,pred_mean,true_mean,precision,recall,f1,accuracy
0,0,0.289257,-0.218795,121.524472,0.488507,0.238023,0.0,0.0,0.0,0.00000
1,5,0.272956,-0.251476,114.676271,0.175042,0.238023,0.0,0.0,0.0,0.96808
2,10,0.238023,-0.064248,100.000000,0.000000,0.238023,0.0,0.0,0.0,1.00000
3,20,0.238023,-0.064248,100.000000,0.000000,0.238023,0.0,0.0,0.0,1.00000
4,30,0.238023,-0.064248,100.000000,0.000000,0.238023,0.0,0.0,0.0,1.00000
5,50,0.238023,-0.064248,100.000000,0.000000,0.238023,0.0,0.0,0.0,1.00000
6,75,0.238023,-0.064248,100.000000,0.000000,0.238023,0.0,0.0,0.0,1.00000
7,100,0.238023,-0.064248,100.000000,0.000000,0.238023,0.0,0.0,0.0,1.00000


In [85]:
balanced_metrics = evaluate(model, balanced_holdout_loader, DEVICE, target_meta)
src_full = evaluate(model, full_holdout_loader, DEVICE, target_meta)

print("=" * 80)
print("CALIBRATED RESULTS")
print("=" * 80)
print(f"Appliance         : {APPLIANCE}")
print(f"Target building   : {TARGET_BUILDING}")
print(f"Calibration ratio : {CALIBRATION_RATIO}")

print("\n[BUILDING_04 BALANCED HOLDOUT]")
for k, v in balanced_metrics.items():
    print(f"{k} : {v}")

print("\n[BUILDING_04 FULL HOLDOUT]")
for k, v in src_full.items():
    print(f"{k} : {v}")

CALIBRATED RESULTS
Appliance         : washing_machine
Target building   : building_04
Calibration ratio : 0.05

[BUILDING_04 BALANCED HOLDOUT]
mae : 0.5267598032951355
r2 : 0.7284488503258881
rel_mae_pct : 22.690194465454923
pred_mean : 2.522803783416748
true_mean : 2.32153058052063
num_samples : 66040

[BUILDING_04 FULL HOLDOUT]
mae : 0.28912121057510376
r2 : -0.22233659546272788
rel_mae_pct : 121.69130795008313
pred_mean : 0.4888229966163635
true_mean : 0.23758575320243835
num_samples : 15527166
